**NEED OF VECTOR STORE**

zrurt kyuu h?
imdb data toh data hona chahie movies ka to ek backend and then frontend ko fetch
if new feature: recommender system based on new genre, actor, director
but issme flaww h ekk.......

ki alg alg movies same dedegaa based on actor, director, genree......ya same bhi show ni hoegi so no quality recommendation
so new approach rather than matching genre u need to match plot...hrr film kaa plot 2000,3000 aisa kchhh
aisa system jo plot ko check kree, score kree and thenn dkheee

ye kaam vse to tough h but dl me embeddings se hoskta h : text ka semantic meaning into numbers (vectors) many dimension
now numbers ke beech similarity aasan h.....toh jse 512 dim to mne multi dim me plot krdiaa and then check krugaa konse 2 vectors h jitna angle bhtt hi kamm h and hence that will be considered as correct movie

but build krne jaoge to dikkt aegi:
challenge: to generate embedding vectors, storage ka issue (not stored in mysql ya relational kyuki inme similarity calculate ni krr paegaa vrna), semantic search..if hrr movie se similarity score to brr brr computation cost....intelligent semantic search meaninggg

so in sabko store krtaa h vector stores and bht neededd

so what is vector store?
system designed to store and retrieve data rep as numerical vectors
features:
1) storing: vector ke sath, metadata jse id, naam...chahe in memory (ram) me store krow ya on drive (harddrive) persistent
2) similarity search: compare to query vector
3) indexing: to fast searching concept...a data structure ki high dim method me fast retrieval
             for eg cluster krdoww, cluster me baatdow....now 10 clusters ka avg so 10 centroid and then query vector ka similarity with similarity score to focus on just cluster 3...
             hence no 10 lakh comparison bht techniques h eseeee
4) crud applications bhi krr skte ho

use cases: recommender system, semantic search, rag, image/multimedia search
basically jahaa bhi vectors kaa kaam isliee use hote h

vector store vs vector db?
store means system jo storage and retrieval ka kam deta h eg FAiss...but if more feature add jse distributed architecture, backup and restore, concurrency, acid and auth etc aisaa kchhh too ismee bolege vector db eg milvus, Qdrant
every vd is vs but not every vs is vd


how vector stores in langchain?
vector store supported : faiss, pinecone, chroma, qdrant in sbke wrappers available and same method signature

from_documents/from_texts to create, add_documents/add_texts and similarity_search metadata based filtering ek ko htake dusra to noo changee in thatt

now we will use chromaaadb

chrome vector store:
lightweight open sourced...vector db h vsee but lightweight bhi h....bw me aata h
data hierarchy: tenant and then multipledb and then multiple db me collection and then multiple collection....(embedding vector and metadata)

do on colab: install lib, import, Document bnadoww: page_content and metadata store krlow...and docs me store....vector store form chroma object: embedding func, presist_directory (kis location me store), collection_name run hote hi sqlite3 file so can be downloaded and used anyelse where

vector_store.add_documents(docs) id bnjatii h and khudki id bhi passed
.get se bhi stored...

now similarity_search.....query=[] k=...kitne storee krnaa h so wow itna easyy
also function similarity_search_with_score: jitna kam utna accha ki distance btata h
also metadata filtering se store: filters leliaaa

update existing document...update_doc_id (same id, updated doc)
delete bhi krrskte ho: kitni bhi id provided

hw...fiuss

In [ ]:
# Install libraries
!pip install -qU langchain langchain-community chromadb langchain-google-genai

# Imports
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_core.documents import Document
from google.colab import userdata

# Gemini API key
GOOGLE_API_KEY = userdata.get("GOOGLE_API_KEY")

# Embedding model
embedding_model = GoogleGenerativeAIEmbeddings(
    model="gemini-embedding-001",
    google_api_key=GOOGLE_API_KEY
)

# ---------------- CREATE DOCUMENTS ----------------

doc1 = Document(
    page_content="""
    Inception is a science fiction movie directed by Christopher Nolan.
    The story revolves around dream invasion where thieves enter people's minds
    to steal or plant ideas. Leonardo DiCaprio plays the lead role.
    """,
    metadata={
        "genre": "Sci-Fi",
        "director": "Christopher Nolan",
        "year": 2010
    }
)

doc2 = Document(
    page_content="""
    Interstellar is a science fiction movie about space exploration.
    A group of astronauts travel through a wormhole to find a new habitable planet
    to save humanity. The movie explores black holes, relativity, and time dilation.
    """,
    metadata={
        "genre": "Sci-Fi",
        "director": "Christopher Nolan",
        "year": 2014
    }
)

doc3 = Document(
    page_content="""
    The Dark Knight is an action superhero film featuring Batman and Joker.
    Joker spreads chaos in Gotham City while Batman tries to stop him.
    Heath Ledger's performance as Joker won critical acclaim.
    """,
    metadata={
        "genre": "Action",
        "director": "Christopher Nolan",
        "year": 2008
    }
)

doc4 = Document(
    page_content="""
    3 Idiots is a comedy-drama Bollywood film about engineering students.
    It focuses on friendship, education pressure, and following passion.
    Aamir Khan played the role of Rancho.
    """,
    metadata={
        "genre": "Drama",
        "director": "Rajkumar Hirani",
        "year": 2009
    }
)

doc5 = Document(
    page_content="""
    Dangal is a sports drama movie based on wrestler Mahavir Singh Phogat.
    It tells the inspiring story of training his daughters to become world-class wrestlers.
    """,
    metadata={
        "genre": "Sports",
        "director": "Nitesh Tiwari",
        "year": 2016
    }
)

docs = [doc1, doc2, doc3, doc4, doc5]

# ---------------- CREATE VECTOR DB ----------------

vector_store = Chroma(
    collection_name="movies_db",
    embedding_function=embedding_model,
    persist_directory="movie1_chroma_db"
)

# ---------------- ADD DOCUMENTS ----------------

doc_ids = vector_store.add_documents(docs)

print("Documents Added!")
print("IDs:", doc_ids)

# ---------------- VIEW DOCUMENTS ----------------

print("\n===== ALL DOCUMENTS =====")
all_docs = vector_store.get(include=["documents", "metadatas"])

for i in range(len(all_docs["documents"])):
    print(f"\nMovie {i+1}")
    print("Content:", all_docs["documents"][i])
    print("Metadata:", all_docs["metadatas"][i])

# ---------------- SIMILARITY SEARCH ----------------

print("\n===== SIMILARITY SEARCH =====")

results = vector_store.similarity_search(
    query="Movie about space and science",
    k=2
)

for i, result in enumerate(results):
    print(f"\nResult {i+1}")
    print(result.page_content)

# ---------------- SEARCH WITH SCORE ----------------

print("\n===== SEARCH WITH SCORE =====")

results_score = vector_store.similarity_search_with_score(
    query="Movie involving dreams",
    k=2
)

for doc, score in results_score:
    print("\nDocument:")
    print(doc.page_content)
    print("Similarity Score:", score)

# ---------------- FILTERING ----------------

print("\n===== SCI-FI MOVIES =====")

# ---------------- METADATA FILTER ----------------

print("\n===== SCI-FI MOVIES =====")

filtered_docs = vector_store.similarity_search(
    query="science fiction movie",
    filter={"genre": "Sci-Fi"}
)

for doc in filtered_docs:
    print(doc.page_content)
for doc in filtered_docs:
    print(doc.page_content)

# ---------------- UPDATE DOCUMENT ----------------

updated_doc = Document(
    page_content="""
    Interstellar is a famous Christopher Nolan sci-fi movie.
    It explores wormholes, black holes, space travel,
    relativity, and survival of humanity.
    """,
    metadata={
        "genre": "Sci-Fi",
        "director": "Christopher Nolan",
        "year": 2014
    }
)

vector_store.update_document(
    document_id=doc_ids[1],
    document=updated_doc
)

print("\nInterstellar document updated!")

# ---------------- DELETE DOCUMENT ----------------

vector_store.delete(ids=[doc_ids[4]])

print("\nDangal deleted!")

# ---------------- FINAL DOCUMENTS ----------------

print("\n===== FINAL DOCUMENTS =====")

final_docs = vector_store.get(include=["documents"])

for doc in final_docs["documents"]:
    print("\n", doc)

Documents Added!
IDs: ['002655f4-59b3-4085-b1fd-c5af23eaf7b6', 'b347b32b-49c0-4e8d-92a5-e4acbf9acc36', '5881aec8-3440-4b16-b2bb-70d6e854812c', 'ed0ad7b8-ee0b-43bb-a111-c534f9fc155e', '034d97c5-a237-4e2e-ba90-6b7d4548a8ba']

===== ALL DOCUMENTS =====

Movie 1
Content: 
    Inception is a science fiction movie directed by Christopher Nolan.
    The story revolves around dream invasion where thieves enter people's minds
    to steal or plant ideas. Leonardo DiCaprio plays the lead role.
    
Metadata: {'director': 'Christopher Nolan', 'genre': 'Sci-Fi', 'year': 2010}

Movie 2
Content: 
    Interstellar is a science fiction movie about space exploration.
    A group of astronauts travel through a wormhole to find a new habitable planet
    to save humanity. The movie explores black holes, relativity, and time dilation.
    
Metadata: {'year': 2014, 'director': 'Christopher Nolan', 'genre': 'Sci-Fi'}

Movie 3
Content: 
    The Dark Knight is an action superhero film featuring Batman and Jok